# LLM Training Loss Analysis
**Model:** e256 | **Layers:** 1, 2, 4, 8 | **Steps:** 0 – 9500

This notebook visualizes training and validation loss across different layer configurations.

## 0. Setup & Data Loading

In [ ]:
# Install required libraries (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'scipy', 'pandas', 'numpy'])
print('All dependencies installed.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.ndimage import uniform_filter1d

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# --- Load CSVs ---
files = {
    'L1': 'log_e256_l1.csv',
    'L2': 'log_e256_l2.csv',
    'L4': 'log_e256_l4.csv',
    'L8': 'log_e256_l8.csv',
}

data = {}
for label, fname in files.items():
    df = pd.read_csv(fname)
    df.columns = df.columns.str.strip()
    data[label] = df
    print(f"{label}: {len(df)} rows | steps {df['step'].min()} → {df['step'].max()}")

COLORS = {'L1': '#e63946', 'L2': '#f4a261', 'L4': '#2a9d8f', 'L8': '#457b9d'}
LAYERS = list(data.keys())

---
## Plot 1 — Training Loss Curves (All Layers)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, df in data.items():
    ax.plot(df['step'], df['train_loss'], label=label, color=COLORS[label], linewidth=2)

ax.set_title('Training Loss — All Layers', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Train Loss')
ax.legend(title='Layers')
plt.tight_layout()
plt.show()

---
## Plot 2 — Validation Loss Curves (All Layers)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, df in data.items():
    ax.plot(df['step'], df['val_loss'], label=label, color=COLORS[label], linewidth=2, linestyle='--')

ax.set_title('Validation Loss — All Layers', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Val Loss')
ax.legend(title='Layers')
plt.tight_layout()
plt.show()

---
## Plot 3 — Train vs Val Loss per Layer (4 Subplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flatten()

for i, (label, df) in enumerate(data.items()):
    ax = axes[i]
    ax.plot(df['step'], df['train_loss'], label='Train', color=COLORS[label], linewidth=2)
    ax.plot(df['step'], df['val_loss'],   label='Val',   color=COLORS[label], linewidth=2,
            linestyle='--', alpha=0.7)
    ax.set_title(f'{label} — Train vs Val', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend()

fig.suptitle('Train vs Validation Loss per Layer', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Plot 4 — Generalization Gap (Val − Train) per Layer

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, df in data.items():
    gap = df['val_loss'] - df['train_loss']
    ax.plot(df['step'], gap, label=label, color=COLORS[label], linewidth=2)

ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_title('Generalization Gap (Val − Train Loss)', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Gap')
ax.legend(title='Layers')
plt.tight_layout()
plt.show()

---
## Plot 5 — Log-Scale Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, df in data.items():
    axes[0].semilogy(df['step'], df['train_loss'], label=label, color=COLORS[label], linewidth=2)
    axes[1].semilogy(df['step'], df['val_loss'],   label=label, color=COLORS[label], linewidth=2, linestyle='--')

for ax, title in zip(axes, ['Train Loss (log scale)', 'Val Loss (log scale)']):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss (log)')
    ax.legend(title='Layers')

fig.suptitle('Log-Scale Loss Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot 6 — Smoothed Loss Curves (Moving Average)

In [ ]:
WINDOW = 3  # adjust smoothing window

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, df in data.items():
    s_train = uniform_filter1d(df['train_loss'], size=WINDOW)
    s_val   = uniform_filter1d(df['val_loss'],   size=WINDOW)
    axes[0].plot(df['step'], s_train, label=label, color=COLORS[label], linewidth=2)
    axes[1].plot(df['step'], s_val,   label=label, color=COLORS[label], linewidth=2, linestyle='--')

for ax, title in zip(axes, [f'Smoothed Train Loss (w={WINDOW})', f'Smoothed Val Loss (w={WINDOW})']):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(title='Layers')

fig.suptitle('Smoothed Loss Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot 7 — Loss Reduction Rate (Step-over-Step Delta)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, df in data.items():
    d_train = -np.diff(df['train_loss'].values)
    d_val   = -np.diff(df['val_loss'].values)
    steps_mid = df['step'].values[1:]
    axes[0].plot(steps_mid, d_train, label=label, color=COLORS[label], linewidth=1.5)
    axes[1].plot(steps_mid, d_val,   label=label, color=COLORS[label], linewidth=1.5, linestyle='--')

for ax, title in zip(axes, ['Train Loss Reduction Rate', 'Val Loss Reduction Rate']):
    ax.axhline(0, color='black', linewidth=0.7, linestyle=':')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('−ΔLoss')
    ax.legend(title='Layers')

fig.suptitle('Loss Reduction Rate per Step Interval', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot 8 — Final Loss Bar Chart (Step 9500)

In [ ]:
labels = LAYERS
final_train = [data[l]['train_loss'].iloc[-1] for l in labels]
final_val   = [data[l]['val_loss'].iloc[-1]   for l in labels]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, final_train, w, label='Train', color=[COLORS[l] for l in labels], alpha=0.9)
bars2 = ax.bar(x + w/2, final_val,   w, label='Val',   color=[COLORS[l] for l in labels], alpha=0.5,
               edgecolor=[COLORS[l] for l in labels], linewidth=1.5)

ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Final Train & Val Loss by Layer (last step)', fontsize=14, fontweight='bold')
ax.set_xlabel('Layer Config')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

---
## Plot 9 — Loss at Key Checkpoints (Grouped Bar)

In [ ]:
CHECKPOINTS = [0, 1000, 2500, 5000, 7500, 9500]

def get_loss_at(df, step, col):
    row = df[df['step'] == step]
    return row[col].values[0] if len(row) else np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train Loss', 'Val Loss']):
    matrix = np.array([[get_loss_at(data[l], s, col) for s in CHECKPOINTS] for l in LAYERS])
    x = np.arange(len(CHECKPOINTS))
    width = 0.18
    for i, label in enumerate(LAYERS):
        ax.bar(x + i*width, matrix[i], width, label=label, color=COLORS[label], alpha=0.85)
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(CHECKPOINTS)
    ax.set_title(f'{title} at Key Checkpoints', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(title='Layers')

plt.tight_layout()
plt.show()

---
## Plot 10 — Steps to Reach Loss Thresholds

In [ ]:
THRESHOLDS = [8.0, 7.0, 6.5, 6.0, 5.8]

def steps_to_threshold(df, col, threshold):
    hits = df[df[col] <= threshold]
    return hits['step'].iloc[0] if len(hits) else np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train', 'Val']):
    for label, df in data.items():
        steps = [steps_to_threshold(df, col, t) for t in THRESHOLDS]
        ax.plot(THRESHOLDS, steps, marker='o', label=label, color=COLORS[label], linewidth=2)
    ax.invert_xaxis()
    ax.set_title(f'Steps to Reach {title} Threshold', fontweight='bold')
    ax.set_xlabel('Loss Threshold')
    ax.set_ylabel('Steps Required')
    ax.legend(title='Layers')

plt.tight_layout()
plt.show()

---
## Plot 11 — Loss Heatmap (Layers × Checkpoints)

In [ ]:
import matplotlib.colors as mcolors

CHECKPOINTS_HM = [0, 500, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 9500]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train Loss', 'Val Loss']):
    matrix = np.array([[get_loss_at(data[l], s, col) for s in CHECKPOINTS_HM] for l in LAYERS])
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn_r')
    ax.set_xticks(range(len(CHECKPOINTS_HM)))
    ax.set_xticklabels(CHECKPOINTS_HM, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(LAYERS)))
    ax.set_yticklabels(LAYERS)
    ax.set_title(f'{title} Heatmap', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Layer')
    for i in range(len(LAYERS)):
        for j in range(len(CHECKPOINTS_HM)):
            ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=6.5, color='black')
    plt.colorbar(im, ax=ax, label='Loss')

plt.tight_layout()
plt.show()

---
## Plot 12 — Area Under the Loss Curve (AUC) — Training Efficiency

In [ ]:
labels = LAYERS
auc_train = [np.trapz(data[l]['train_loss'], data[l]['step']) for l in labels]
auc_val   = [np.trapz(data[l]['val_loss'],   data[l]['step']) for l in labels]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, auc_train, w, label='Train AUC', color=[COLORS[l] for l in labels], alpha=0.9)
b2 = ax.bar(x + w/2, auc_val,   w, label='Val AUC',   color=[COLORS[l] for l in labels], alpha=0.5,
            edgecolor=[COLORS[l] for l in labels], linewidth=1.5)
ax.bar_label(b1, fmt='%.0f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.0f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Area Under the Loss Curve (AUC) — Lower is Better', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Config')
ax.set_ylabel('AUC (loss × steps)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Plot 13 — Val/Train Loss Ratio Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, df in data.items():
    ratio = df['val_loss'] / df['train_loss']
    ax.plot(df['step'], ratio, label=label, color=COLORS[label], linewidth=2)

ax.axhline(1.0, color='black', linewidth=0.8, linestyle=':', label='ratio = 1')
ax.set_title('Val / Train Loss Ratio Over Steps', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Val / Train')
ax.legend(title='Layers')
plt.tight_layout()
plt.show()

---
---
# Part 2 — Embedding Dimension Analysis (4 Layers Fixed)
**Embedding dims:** 128, 256, 512, 1024 | **Layers fixed at 4**

## E-Setup — Load Embedding Dim Data

In [ ]:
efiles = {
    'E128':  'log_e128_l4.csv',
    'E256':  'log_e256_l4__1_.csv',
    'E512':  'log_e512_l4.csv',
    'E1024': 'log_e1024_l4.csv',
}

edata = {}
for label, fname in efiles.items():
    df = pd.read_csv(fname)
    df.columns = df.columns.str.strip()
    edata[label] = df
    print(f"{label}: {len(df)} rows | steps {df['step'].min()} -> {df['step'].max()}")

ECOLORS = {'E128': '#9b2226', 'E256': '#ca6702', 'E512': '#0a9396', 'E1024': '#005f73'}
EDIMS   = [128, 256, 512, 1024]
ELABELS = list(edata.keys())

---
## Plot E1 — Training Loss Curves (All Embedding Dims)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, df in edata.items():
    ax.plot(df['step'], df['train_loss'], label=label, color=ECOLORS[label], linewidth=2)
ax.set_title('Training Loss — All Embedding Dims (L4 fixed)', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Train Loss')
ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E2 — Validation Loss Curves (All Embedding Dims)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, df in edata.items():
    ax.plot(df['step'], df['val_loss'], label=label, color=ECOLORS[label], linewidth=2, linestyle='--')
ax.set_title('Validation Loss — All Embedding Dims (L4 fixed)', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Val Loss')
ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E3 — Train vs Val per Embedding Dim (4 Subplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flatten()
for i, (label, df) in enumerate(edata.items()):
    ax = axes[i]
    ax.plot(df['step'], df['train_loss'], label='Train', color=ECOLORS[label], linewidth=2)
    ax.plot(df['step'], df['val_loss'],   label='Val',   color=ECOLORS[label], linewidth=2, linestyle='--', alpha=0.7)
    ax.set_title(f'{label} — Train vs Val', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend()
fig.suptitle('Train vs Val Loss per Embedding Dim', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Plot E4 — Generalization Gap by Embedding Dim

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, df in edata.items():
    gap = df['val_loss'] - df['train_loss']
    ax.plot(df['step'], gap, label=label, color=ECOLORS[label], linewidth=2)
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_title('Generalization Gap (Val − Train) by Embedding Dim', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Gap')
ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E5 — Final Loss Bar Chart by Embedding Dim

In [ ]:
final_train_e = [edata[l]['train_loss'].iloc[-1] for l in ELABELS]
final_val_e   = [edata[l]['val_loss'].iloc[-1]   for l in ELABELS]
x = np.arange(len(ELABELS))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, final_train_e, w, label='Train', color=[ECOLORS[l] for l in ELABELS], alpha=0.9)
b2 = ax.bar(x + w/2, final_val_e,   w, label='Val',   color=[ECOLORS[l] for l in ELABELS], alpha=0.5,
            edgecolor=[ECOLORS[l] for l in ELABELS], linewidth=1.5)
ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(ELABELS)
ax.set_title('Final Train & Val Loss by Embedding Dim (last step)', fontsize=13, fontweight='bold')
ax.set_xlabel('Embedding Dim')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

---
## Plot E6 — Loss at Key Checkpoints by Embedding Dim

In [ ]:
CHECKPOINTS_E = [0, 1000, 2500, 5000, 7500, 9500]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train Loss', 'Val Loss']):
    matrix = np.array([[get_loss_at(edata[l], s, col) for s in CHECKPOINTS_E] for l in ELABELS])
    x = np.arange(len(CHECKPOINTS_E))
    width = 0.18
    for i, label in enumerate(ELABELS):
        ax.bar(x + i*width, matrix[i], width, label=label, color=ECOLORS[label], alpha=0.85)
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(CHECKPOINTS_E)
    ax.set_title(f'{title} at Key Checkpoints', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E7 — Steps to Reach Loss Thresholds by Embedding Dim

In [ ]:
THRESHOLDS_E = [8.0, 7.0, 6.5, 6.0, 5.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train', 'Val']):
    for label, df in edata.items():
        steps = [steps_to_threshold(df, col, t) for t in THRESHOLDS_E]
        ax.plot(THRESHOLDS_E, steps, marker='o', label=label, color=ECOLORS[label], linewidth=2)
    ax.invert_xaxis()
    ax.set_title(f'Steps to Reach {title} Threshold', fontweight='bold')
    ax.set_xlabel('Loss Threshold')
    ax.set_ylabel('Steps Required')
    ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E8 — AUC (Training Efficiency) by Embedding Dim

In [ ]:
auc_train_e = [np.trapz(edata[l]['train_loss'], edata[l]['step']) for l in ELABELS]
auc_val_e   = [np.trapz(edata[l]['val_loss'],   edata[l]['step']) for l in ELABELS]
x = np.arange(len(ELABELS))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, auc_train_e, w, label='Train AUC', color=[ECOLORS[l] for l in ELABELS], alpha=0.9)
b2 = ax.bar(x + w/2, auc_val_e,   w, label='Val AUC',   color=[ECOLORS[l] for l in ELABELS], alpha=0.5,
            edgecolor=[ECOLORS[l] for l in ELABELS], linewidth=1.5)
ax.bar_label(b1, fmt='%.0f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.0f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(ELABELS)
ax.set_title('Area Under Loss Curve by Embedding Dim — Lower is Better', fontsize=13, fontweight='bold')
ax.set_xlabel('Embedding Dim')
ax.set_ylabel('AUC (loss x steps)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Plot E9 — Val/Train Ratio by Embedding Dim

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, df in edata.items():
    ratio = df['val_loss'] / df['train_loss']
    ax.plot(df['step'], ratio, label=label, color=ECOLORS[label], linewidth=2)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle=':', label='ratio = 1')
ax.set_title('Val / Train Loss Ratio by Embedding Dim', fontsize=14, fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Val / Train')
ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E10 — Scaling Law: Final Loss vs Embedding Dim (log2 scale)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
final_t = [edata[l]['train_loss'].iloc[-1] for l in ELABELS]
final_v = [edata[l]['val_loss'].iloc[-1]   for l in ELABELS]
ax.plot(EDIMS, final_t, 'o-', color='#0a9396', linewidth=2, markersize=8, label='Train')
ax.plot(EDIMS, final_v, 's--', color='#ca6702', linewidth=2, markersize=8, label='Val')
for dim, t, v in zip(EDIMS, final_t, final_v):
    ax.annotate(f'{t:.3f}', (dim, t), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
    ax.annotate(f'{v:.3f}', (dim, v), textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8)
ax.set_xscale('log', base=2)
ax.set_xticks(EDIMS)
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.set_title('Scaling Law: Final Loss vs Embedding Dim (log2 x-axis)', fontsize=13, fontweight='bold')
ax.set_xlabel('Embedding Dimension (log2 scale)')
ax.set_ylabel('Final Loss')
ax.legend()
plt.tight_layout()
plt.show()

---
## Plot E11 — Loss vs Embedding Dim at Fixed Steps

In [ ]:
FIXED_STEPS = [500, 1000, 2500, 5000, 9500]
STEP_COLORS = ['#e63946', '#f4a261', '#2a9d8f', '#457b9d', '#6a4c93']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train Loss', 'Val Loss']):
    for s, c in zip(FIXED_STEPS, STEP_COLORS):
        vals = [get_loss_at(edata[l], s, col) for l in ELABELS]
        ax.plot(EDIMS, vals, 'o-', color=c, linewidth=2, markersize=7, label=f'step {s}')
    ax.set_xscale('log', base=2)
    ax.set_xticks(EDIMS)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    ax.set_title(f'{title} vs Embedding Dim at Fixed Steps', fontweight='bold')
    ax.set_xlabel('Embedding Dimension')
    ax.set_ylabel('Loss')
    ax.legend(title='Step', fontsize=8)
plt.tight_layout()
plt.show()

---
## Plot E12 — Relative Improvement over E128 (Baseline)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
base_label = 'E128'
for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train', 'Val']):
    base = edata[base_label][col].values
    for label, df in edata.items():
        if label == base_label:
            continue
        rel_improvement = (base - df[col].values) / base * 100
        ax.plot(df['step'], rel_improvement, label=label, color=ECOLORS[label], linewidth=2)
    ax.axhline(0, color='black', linewidth=0.7, linestyle=':')
    ax.set_title(f'{title} Loss: % Improvement over E128', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('% Improvement over E128')
    ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot E13 — Embedding Dim Loss Heatmap (Dim x Checkpoints)

In [ ]:
CHECKPOINTS_EHM = [0, 500, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 9500]
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train Loss', 'Val Loss']):
    matrix = np.array([[get_loss_at(edata[l], s, col) for s in CHECKPOINTS_EHM] for l in ELABELS])
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn_r')
    ax.set_xticks(range(len(CHECKPOINTS_EHM)))
    ax.set_xticklabels(CHECKPOINTS_EHM, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(ELABELS)))
    ax.set_yticklabels(ELABELS)
    ax.set_title(f'{title} Heatmap — Embed Dim x Step', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Embedding Dim')
    for i in range(len(ELABELS)):
        for j in range(len(CHECKPOINTS_EHM)):
            ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=6.5, color='black')
    plt.colorbar(im, ax=ax, label='Loss')
plt.tight_layout()
plt.show()

---
---
# Part 3 — Cross-Axis Analysis (Layers x Embedding Dims)
Combining both datasets at the final step.

---
## Plot C1 — Cross Heatmap: Layers x Embedding Dim (Final Loss)

In [ ]:
layer_nums = [1, 2, 4, 8]
layer_keys = ['L1', 'L2', 'L4', 'L8']
edim_nums  = [128, 256, 512, 1024]
edim_keys  = ['E128', 'E256', 'E512', 'E1024']

mat_train = np.full((len(layer_nums), len(edim_nums)), np.nan)
mat_val   = np.full((len(layer_nums), len(edim_nums)), np.nan)

e256_col = edim_nums.index(256)
for r, lk in enumerate(layer_keys):
    mat_train[r, e256_col] = data[lk]['train_loss'].iloc[-1]
    mat_val[r,   e256_col] = data[lk]['val_loss'].iloc[-1]

l4_row = layer_nums.index(4)
for c, ek in enumerate(edim_keys):
    mat_train[l4_row, c] = edata[ek]['train_loss'].iloc[-1]
    mat_val[l4_row,   c] = edata[ek]['val_loss'].iloc[-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, mat, title in zip(axes, [mat_train, mat_val], ['Final Train Loss', 'Final Val Loss']):
    masked = np.ma.masked_invalid(mat)
    im = ax.imshow(masked, aspect='auto', cmap='RdYlGn_r')
    ax.set_xticks(range(len(edim_nums)))
    ax.set_xticklabels(edim_nums)
    ax.set_yticks(range(len(layer_nums)))
    ax.set_yticklabels(layer_nums)
    ax.set_xlabel('Embedding Dim')
    ax.set_ylabel('Num Layers')
    ax.set_title(f'{title}: Layers x Embed Dim', fontweight='bold')
    for i in range(len(layer_nums)):
        for j in range(len(edim_nums)):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f'{mat[i,j]:.3f}', ha='center', va='center', fontsize=8, color='black')
            else:
                ax.text(j, i, 'N/A', ha='center', va='center', fontsize=7, color='grey')
    plt.colorbar(im, ax=ax, label='Loss')
plt.tight_layout()
plt.show()

---
## Plot C2 — Scaling Curves: Final Loss vs Layers AND vs Embed Dim

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
layer_final_train = [data[lk]['train_loss'].iloc[-1] for lk in layer_keys]
layer_final_val   = [data[lk]['val_loss'].iloc[-1]   for lk in layer_keys]
ax.plot(layer_nums, layer_final_train, 'o-', color='#0a9396', linewidth=2, markersize=8, label='Train')
ax.plot(layer_nums, layer_final_val,   's--', color='#ca6702', linewidth=2, markersize=8, label='Val')
ax.set_xscale('log', base=2)
ax.set_xticks(layer_nums)
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.set_title('Final Loss vs Num Layers (E256 fixed)', fontweight='bold')
ax.set_xlabel('Num Layers')
ax.set_ylabel('Final Loss')
ax.legend()

ax = axes[1]
edim_final_train = [edata[ek]['train_loss'].iloc[-1] for ek in edim_keys]
edim_final_val   = [edata[ek]['val_loss'].iloc[-1]   for ek in edim_keys]
ax.plot(edim_nums, edim_final_train, 'o-', color='#0a9396', linewidth=2, markersize=8, label='Train')
ax.plot(edim_nums, edim_final_val,   's--', color='#ca6702', linewidth=2, markersize=8, label='Val')
ax.set_xscale('log', base=2)
ax.set_xticks(edim_nums)
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.set_title('Final Loss vs Embedding Dim (L4 fixed)', fontweight='bold')
ax.set_xlabel('Embedding Dim')
ax.set_ylabel('Final Loss')
ax.legend()

fig.suptitle('Scaling Curves: Layers vs Embedding Dim', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
---
# Part 4 — Evaluation Metrics Analysis (E256, Layers 1/2/4/8)
**Metrics:** Language_Accuracy, Distinct_1, Distinct_2, Overlap, Repetition, Perplexity  
**X-axis:** Sampling Temperature (0.5 → 1.2)  
**Layers:** L1, L2, L4, L8 (embedding dim fixed at 256)

## M-Setup — Load Eval Metric Data

In [ ]:
mfiles = {
    'L1': 'eval_e256_l1.csv',
    'L2': 'eval_e256_l2.csv',
    'L4': 'eval_e256_l4.csv',
    'L8': 'eval_e256_l8.csv',
}

mdata = {}
for label, fname in mfiles.items():
    df = pd.read_csv(fname)
    df.columns = df.columns.str.strip()
    mdata[label] = df
    print(f"{label}: {df.shape} | temps: {df['Temperature'].tolist()}")

MCOLORS  = {'L1': '#e63946', 'L2': '#f4a261', 'L4': '#2a9d8f', 'L8': '#457b9d'}
MLABELS  = list(mdata.keys())
METRICS  = ['Language_Accuracy', 'Distinct_1', 'Distinct_2', 'Overlap', 'Repetition', 'Perplexity']
# Higher-is-better flags for annotations
HIGHER_BETTER = {
    'Language_Accuracy': True, 'Distinct_1': True, 'Distinct_2': True,
    'Overlap': False, 'Repetition': False, 'Perplexity': False
}
TEMPS = mdata['L1']['Temperature'].tolist()

---
## Plot M1 — All Metrics vs Temperature (One Subplot per Metric)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True)
axes = axes.flatten()

for i, metric in enumerate(METRICS):
    ax = axes[i]
    for label, df in mdata.items():
        ax.plot(df['Temperature'], df[metric], marker='o', label=label,
                color=MCOLORS[label], linewidth=2, markersize=6)
    direction = 'higher better' if HIGHER_BETTER[metric] else 'lower better'
    ax.set_title(f'{metric}  ({direction})', fontweight='bold')
    ax.set_xlabel('Temperature')
    ax.set_ylabel(metric)
    ax.legend(title='Layers', fontsize=8)

fig.suptitle('Eval Metrics vs Sampling Temperature — All Layers', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot M2 — Perplexity by Layer (Temperature-Independent)

In [ ]:
# Perplexity is identical across temperatures per layer — use first row
perp_vals = [mdata[l]['Perplexity'].iloc[0] for l in MLABELS]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(MLABELS, perp_vals, color=[MCOLORS[l] for l in MLABELS], alpha=0.88, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=10, fontweight='bold')
ax.set_title('Perplexity by Layer Count (lower is better)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Config')
ax.set_ylabel('Perplexity')
ax.set_ylim(0, max(perp_vals) * 1.15)
plt.tight_layout()
plt.show()

---
## Plot M3 — Diversity vs Repetition Tradeoff (Scatter per Temperature)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

markers = ['o', 's', '^', 'D']  # one per temperature
for label, df in mdata.items():
    for i, (_, row) in enumerate(df.iterrows()):
        ax.scatter(row['Repetition'], row['Distinct_2'],
                   color=MCOLORS[label], marker=markers[i], s=120,
                   label=f"{label} T={row['Temperature']}" if i == 0 else None,
                   zorder=3)
        ax.annotate(f"T={row['Temperature']}", (row['Repetition'], row['Distinct_2']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7, color=MCOLORS[label])
    # connect dots per layer
    ax.plot(df['Repetition'], df['Distinct_2'], color=MCOLORS[label], linewidth=1, alpha=0.4)

# Legend for layers only
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], color=MCOLORS[l], marker='o', linewidth=2, label=l) for l in MLABELS]
ax.legend(handles=handles, title='Layer')
ax.set_xlabel('Repetition  (lower is better →)')
ax.set_ylabel('Distinct_2  (higher is better ↑)')
ax.set_title('Diversity vs Repetition Tradeoff across Temperature & Layers', fontsize=13, fontweight='bold')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

---
## Plot M4 — Metric Heatmap: Layer × Metric at Each Temperature

In [ ]:
from matplotlib.colors import Normalize
import matplotlib.cm as cm

non_perp = [m for m in METRICS if m != 'Perplexity']
fig, axes = plt.subplots(1, len(TEMPS), figsize=(16, 4), sharey=True)

for ax, temp in zip(axes, TEMPS):
    matrix = np.array([
        [mdata[l].loc[mdata[l]['Temperature'] == temp, m].values[0] for m in non_perp]
        for l in MLABELS
    ])
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(non_perp)))
    ax.set_xticklabels(non_perp, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(MLABELS)))
    ax.set_yticklabels(MLABELS)
    ax.set_title(f'Temp = {temp}', fontweight='bold')
    for i in range(len(MLABELS)):
        for j in range(len(non_perp)):
            ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center', fontsize=7)

fig.suptitle('Layer × Metric Heatmap at Each Temperature (excl. Perplexity)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[-1], label='Value')
plt.tight_layout()
plt.show()

---
## Plot M5 — Per-Layer Heatmap: Metric × Temperature

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)

for ax, label in zip(axes, MLABELS):
    df = mdata[label]
    matrix = np.array([
        [df.loc[df['Temperature'] == t, m].values[0] for t in TEMPS]
        for m in non_perp
    ])
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(TEMPS)))
    ax.set_xticklabels(TEMPS)
    ax.set_yticks(range(len(non_perp)))
    ax.set_yticklabels(non_perp, fontsize=9)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Temperature')
    for i in range(len(non_perp)):
        for j in range(len(TEMPS)):
            ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center', fontsize=7)

fig.suptitle('Metric × Temperature Heatmap per Layer (excl. Perplexity)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[-1], label='Value')
plt.tight_layout()
plt.show()

---
## Plot M6 — Radar Chart: Layer Comparison per Temperature

In [ ]:
# Normalise metrics 0-1 (flip lower-is-better so bigger = better always)
radar_metrics = ['Language_Accuracy', 'Distinct_1', 'Distinct_2', 'Overlap', 'Repetition']
# For radar: invert Overlap & Repetition (lower is better → flip)
INVERT = {'Overlap', 'Repetition'}

N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, axes = plt.subplots(1, len(TEMPS), figsize=(16, 4),
                          subplot_kw=dict(polar=True))

for ax, temp in zip(axes, TEMPS):
    for label, df in mdata.items():
        row = df[df['Temperature'] == temp].iloc[0]
        vals = []
        for m in radar_metrics:
            v = row[m]
            vals.append(1 - v if m in INVERT else v)
        vals += vals[:1]
        ax.plot(angles, vals, color=MCOLORS[label], linewidth=2, label=label)
        ax.fill(angles, vals, color=MCOLORS[label], alpha=0.08)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['Accuracy', 'Distinct1', 'Distinct2', '1-Overlap', '1-Repeat'], fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'Temp {temp}', fontweight='bold', pad=12)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=7)

fig.suptitle('Radar Chart — Layer Quality Profile per Temperature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot M7 — Vocabulary Richness: Distinct-1 vs Distinct-2

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, df in mdata.items():
    sc = ax.scatter(df['Distinct_1'], df['Distinct_2'],
                    c=df['Temperature'], cmap='plasma',
                    s=120, edgecolors=MCOLORS[label], linewidths=2,
                    label=label, zorder=3)
    ax.plot(df['Distinct_1'], df['Distinct_2'], color=MCOLORS[label], linewidth=1, alpha=0.4)
    for _, row in df.iterrows():
        ax.annotate(f"T={row['Temperature']}",
                    (row['Distinct_1'], row['Distinct_2']),
                    textcoords='offset points', xytext=(5, 3), fontsize=7, color=MCOLORS[label])

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Temperature')
handles = [Line2D([0],[0], color=MCOLORS[l], marker='o', linewidth=2, label=l) for l in MLABELS]
ax.legend(handles=handles, title='Layer')
ax.set_xlabel('Distinct-1 (unigram diversity)')
ax.set_ylabel('Distinct-2 (bigram diversity)')
ax.set_title('Vocabulary Richness: Distinct-1 vs Distinct-2 across Temperature & Layers',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot M8 — Composite Quality Score vs Temperature

In [ ]:
# Score = mean of (higher-is-better) + (1 - lower-is-better), excl. Perplexity
def composite_score(row):
    components = [
        row['Language_Accuracy'],
        row['Distinct_1'],
        row['Distinct_2'],
        1 - row['Overlap'],
        1 - row['Repetition'],
    ]
    return np.mean(components)

fig, ax = plt.subplots(figsize=(9, 5))
for label, df in mdata.items():
    scores = df.apply(composite_score, axis=1)
    ax.plot(df['Temperature'], scores, marker='o', label=label,
            color=MCOLORS[label], linewidth=2.5, markersize=8)
    # Annotate best temp
    best_idx = scores.idxmax()
    ax.annotate(f"best", (df['Temperature'].iloc[best_idx], scores.iloc[best_idx]),
                textcoords='offset points', xytext=(4, 6), fontsize=7, color=MCOLORS[label])

ax.set_title('Composite Quality Score vs Temperature (higher is better)', fontsize=13, fontweight='bold')
ax.set_xlabel('Temperature')
ax.set_ylabel('Composite Score')
ax.legend(title='Layers')
plt.tight_layout()
plt.show()

---
## Plot M9 — Best-Temperature Profile per Layer (Bar Chart)

In [ ]:
def best_temp_row(df):
    scores = df.apply(composite_score, axis=1)
    return df.iloc[scores.idxmax()]

best_rows = {l: best_temp_row(mdata[l]) for l in MLABELS}

plot_metrics = ['Distinct_1', 'Distinct_2', 'Overlap', 'Repetition']
x = np.arange(len(plot_metrics))
w = 0.18

fig, ax = plt.subplots(figsize=(11, 5))
for i, label in enumerate(MLABELS):
    row = best_rows[label]
    vals = [row[m] for m in plot_metrics]
    bars = ax.bar(x + i*w, vals, w, label=f"{label} (T={row['Temperature']})",
                  color=MCOLORS[label], alpha=0.88)

ax.set_xticks(x + w*1.5)
ax.set_xticklabels(plot_metrics)
ax.set_title('Metric Values at Best Temperature per Layer', fontsize=13, fontweight='bold')
ax.set_ylabel('Value')
ax.legend(title='Layer (best temp)', fontsize=9)
plt.tight_layout()
plt.show()

---
## Plot M10 — Perplexity vs Diversity (Distinct_2) by Layer

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

perp_vals_m = [mdata[l]['Perplexity'].iloc[0] for l in MLABELS]
d2_at_best  = [best_temp_row(mdata[l])['Distinct_2'] for l in MLABELS]

for i, label in enumerate(MLABELS):
    ax.scatter(perp_vals_m[i], d2_at_best[i], color=MCOLORS[label], s=180, zorder=3,
               edgecolors='white', linewidths=1.5)
    ax.annotate(label, (perp_vals_m[i], d2_at_best[i]),
                textcoords='offset points', xytext=(6, 5), fontsize=11,
                fontweight='bold', color=MCOLORS[label])

ax.set_xlabel('Perplexity  (lower is better ←)')
ax.set_ylabel('Distinct_2 at best temp  (higher is better ↑)')
ax.set_title('Perplexity vs Generation Diversity — Layer Comparison',
             fontsize=13, fontweight='bold')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

---
## Plot M11 — Overlap & Repetition vs Temperature (Side by Side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)

for ax, metric in zip(axes, ['Overlap', 'Repetition']):
    for label, df in mdata.items():
        ax.plot(df['Temperature'], df[metric], marker='o', label=label,
                color=MCOLORS[label], linewidth=2, markersize=7)
    ax.set_title(f'{metric} vs Temperature  (lower is better)', fontweight='bold')
    ax.set_xlabel('Temperature')
    ax.set_ylabel(metric)
    ax.legend(title='Layers')

fig.suptitle('Coherence Metrics: Overlap & Repetition across Temperature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
---
# Part 5 — Eval Metrics: Embedding Dimension Analysis (L4 Fixed)
**Embedding dims:** 128, 256, 512, 1024  |  **Layers fixed at 4**

**Metrics:** Language_Accuracy, Distinct_1, Distinct_2, Overlap, Repetition, Perplexity

## ME-Setup — Load Embedding Dim Eval Data

In [ ]:
mefiles = {
    'E128':  'eval_e128_l4.csv',
    'E256':  'eval_e256_l4__1_.csv',
    'E512':  'eval_e512_l4.csv',
    'E1024': 'eval_e1024_l4.csv',
}

medata = {}
for label, fname in mefiles.items():
    df = pd.read_csv(fname)
    df.columns = df.columns.str.strip()
    medata[label] = df
    print(f"{label}: {df.shape} | temps: {df['Temperature'].tolist()}")

MECOLORS = {'E128': '#9b2226', 'E256': '#ca6702', 'E512': '#0a9396', 'E1024': '#005f73'}
MELABELS = list(medata.keys())
EDIMS_EVAL = [128, 256, 512, 1024]
METRICS    = ['Language_Accuracy', 'Distinct_1', 'Distinct_2', 'Overlap', 'Repetition', 'Perplexity']
HIGHER_BETTER = {
    'Language_Accuracy': True, 'Distinct_1': True, 'Distinct_2': True,
    'Overlap': False, 'Repetition': False, 'Perplexity': False
}
non_perp   = [m for m in METRICS if m != 'Perplexity']
INVERT     = {'Overlap', 'Repetition'}
TEMPS_E    = medata['E128']['Temperature'].tolist()

def composite_score(row):
    return np.mean([
        row['Language_Accuracy'], row['Distinct_1'], row['Distinct_2'],
        1 - row['Overlap'], 1 - row['Repetition']
    ])

def best_temp_row_e(df):
    scores = df.apply(composite_score, axis=1)
    return df.iloc[scores.idxmax()]

---
## Plot ME1 — All Metrics vs Temperature (One Subplot per Metric)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True)
axes = axes.flatten()
for i, metric in enumerate(METRICS):
    ax = axes[i]
    for label, df in medata.items():
        ax.plot(df['Temperature'], df[metric], marker='o', label=label,
                color=MECOLORS[label], linewidth=2, markersize=6)
    direction = 'higher better' if HIGHER_BETTER[metric] else 'lower better'
    ax.set_title(f'{metric}  ({direction})', fontweight='bold')
    ax.set_xlabel('Temperature')
    ax.set_ylabel(metric)
    ax.legend(title='Embed Dim', fontsize=8)
fig.suptitle('Eval Metrics vs Temperature — All Embedding Dims (L4 Fixed)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot ME2 — Perplexity by Embedding Dim (Temperature-Independent)

In [ ]:
perp_e = [medata[l]['Perplexity'].iloc[0] for l in MELABELS]
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(MELABELS, perp_e, color=[MECOLORS[l] for l in MELABELS], alpha=0.88,
               edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=10, fontweight='bold')
ax.set_title('Perplexity by Embedding Dim (lower is better)', fontsize=13, fontweight='bold')
ax.set_xlabel('Embedding Dim')
ax.set_ylabel('Perplexity')
ax.set_ylim(0, max(perp_e) * 1.15)
plt.tight_layout()
plt.show()

---
## Plot ME3 — Diversity vs Repetition Tradeoff across Temperature & Embed Dim

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(9, 6))
markers = ['o', 's', '^', 'D']
for label, df in medata.items():
    for i, (_, row) in enumerate(df.iterrows()):
        ax.scatter(row['Repetition'], row['Distinct_2'],
                   color=MECOLORS[label], marker=markers[i], s=120,
                   zorder=3)
        ax.annotate(f"T={row['Temperature']}", (row['Repetition'], row['Distinct_2']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7, color=MECOLORS[label])
    ax.plot(df['Repetition'], df['Distinct_2'], color=MECOLORS[label], linewidth=1, alpha=0.4)
handles = [Line2D([0],[0], color=MECOLORS[l], marker='o', linewidth=2, label=l) for l in MELABELS]
ax.legend(handles=handles, title='Embed Dim')
ax.set_xlabel('Repetition  (lower is better →)')
ax.set_ylabel('Distinct_2  (higher is better ↑)')
ax.set_title('Diversity vs Repetition Tradeoff — Embedding Dims', fontsize=13, fontweight='bold')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

---
## Plot ME4 — Metric Heatmap: Embed Dim × Metric at Each Temperature

In [ ]:
fig, axes = plt.subplots(1, len(TEMPS_E), figsize=(16, 4), sharey=True)
for ax, temp in zip(axes, TEMPS_E):
    matrix = np.array([
        [medata[l].loc[medata[l]['Temperature'] == temp, m].values[0] for m in non_perp]
        for l in MELABELS
    ])
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(non_perp)))
    ax.set_xticklabels(non_perp, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(MELABELS)))
    ax.set_yticklabels(MELABELS)
    ax.set_title(f'Temp = {temp}', fontweight='bold')
    for i in range(len(MELABELS)):
        for j in range(len(non_perp)):
            ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center', fontsize=7)
fig.suptitle('Embed Dim × Metric Heatmap at Each Temperature (excl. Perplexity)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[-1], label='Value')
plt.tight_layout()
plt.show()

---
## Plot ME5 — Per Embed-Dim Heatmap: Metric × Temperature

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)
for ax, label in zip(axes, MELABELS):
    df = medata[label]
    matrix = np.array([
        [df.loc[df['Temperature'] == t, m].values[0] for t in TEMPS_E]
        for m in non_perp
    ])
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(TEMPS_E)))
    ax.set_xticklabels(TEMPS_E)
    ax.set_yticks(range(len(non_perp)))
    ax.set_yticklabels(non_perp, fontsize=9)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Temperature')
    for i in range(len(non_perp)):
        for j in range(len(TEMPS_E)):
            ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center', fontsize=7)
fig.suptitle('Metric × Temperature Heatmap per Embed Dim (excl. Perplexity)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[-1], label='Value')
plt.tight_layout()
plt.show()

---
## Plot ME6 — Radar Chart: Embed Dim Quality Profile per Temperature

In [ ]:
radar_metrics = ['Language_Accuracy', 'Distinct_1', 'Distinct_2', 'Overlap', 'Repetition']
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]
fig, axes = plt.subplots(1, len(TEMPS_E), figsize=(16, 4), subplot_kw=dict(polar=True))
for ax, temp in zip(axes, TEMPS_E):
    for label, df in medata.items():
        row = df[df['Temperature'] == temp].iloc[0]
        vals = [1 - row[m] if m in INVERT else row[m] for m in radar_metrics]
        vals += vals[:1]
        ax.plot(angles, vals, color=MECOLORS[label], linewidth=2, label=label)
        ax.fill(angles, vals, color=MECOLORS[label], alpha=0.08)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['Accuracy', 'Distinct1', 'Distinct2', '1-Overlap', '1-Repeat'], fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'Temp {temp}', fontweight='bold', pad=12)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)
fig.suptitle('Radar Chart — Embed Dim Quality Profile per Temperature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot ME7 — Vocabulary Richness: Distinct-1 vs Distinct-2

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label, df in medata.items():
    sc = ax.scatter(df['Distinct_1'], df['Distinct_2'],
                    c=df['Temperature'], cmap='plasma',
                    s=120, edgecolors=MECOLORS[label], linewidths=2,
                    label=label, zorder=3)
    ax.plot(df['Distinct_1'], df['Distinct_2'], color=MECOLORS[label], linewidth=1, alpha=0.4)
    for _, row in df.iterrows():
        ax.annotate(f"T={row['Temperature']}", (row['Distinct_1'], row['Distinct_2']),
                    textcoords='offset points', xytext=(5, 3), fontsize=7, color=MECOLORS[label])
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Temperature')
handles = [Line2D([0],[0], color=MECOLORS[l], marker='o', linewidth=2, label=l) for l in MELABELS]
ax.legend(handles=handles, title='Embed Dim')
ax.set_xlabel('Distinct-1 (unigram diversity)')
ax.set_ylabel('Distinct-2 (bigram diversity)')
ax.set_title('Vocabulary Richness: Distinct-1 vs Distinct-2 — Embedding Dims', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot ME8 — Composite Quality Score vs Temperature

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, df in medata.items():
    scores = df.apply(composite_score, axis=1)
    ax.plot(df['Temperature'], scores, marker='o', label=label,
            color=MECOLORS[label], linewidth=2.5, markersize=8)
    best_idx = scores.idxmax()
    ax.annotate('best', (df['Temperature'].iloc[best_idx], scores.iloc[best_idx]),
                textcoords='offset points', xytext=(4, 6), fontsize=7, color=MECOLORS[label])
ax.set_title('Composite Quality Score vs Temperature — Embedding Dims (higher is better)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Temperature')
ax.set_ylabel('Composite Score')
ax.legend(title='Embed Dim')
plt.tight_layout()
plt.show()

---
## Plot ME9 — Best-Temperature Metric Profile per Embed Dim

In [ ]:
best_rows_e = {l: best_temp_row_e(medata[l]) for l in MELABELS}
plot_metrics_e = ['Distinct_1', 'Distinct_2', 'Overlap', 'Repetition']
x = np.arange(len(plot_metrics_e))
w = 0.18
fig, ax = plt.subplots(figsize=(11, 5))
for i, label in enumerate(MELABELS):
    row = best_rows_e[label]
    vals = [row[m] for m in plot_metrics_e]
    ax.bar(x + i*w, vals, w, label=f"{label} (T={row['Temperature']})",
           color=MECOLORS[label], alpha=0.88)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(plot_metrics_e)
ax.set_title('Metric Values at Best Temperature per Embed Dim', fontsize=13, fontweight='bold')
ax.set_ylabel('Value')
ax.legend(title='Embed Dim (best temp)', fontsize=9)
plt.tight_layout()
plt.show()

---
## Plot ME10 — Perplexity vs Diversity (Distinct_2) by Embed Dim

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
perp_e_vals = [medata[l]['Perplexity'].iloc[0] for l in MELABELS]
d2_best_e   = [best_temp_row_e(medata[l])['Distinct_2'] for l in MELABELS]
for i, label in enumerate(MELABELS):
    ax.scatter(perp_e_vals[i], d2_best_e[i], color=MECOLORS[label], s=180, zorder=3,
               edgecolors='white', linewidths=1.5)
    ax.annotate(label, (perp_e_vals[i], d2_best_e[i]),
                textcoords='offset points', xytext=(6, 5), fontsize=11,
                fontweight='bold', color=MECOLORS[label])
ax.set_xlabel('Perplexity  (lower is better ←)')
ax.set_ylabel('Distinct_2 at best temp  (higher is better ↑)')
ax.set_title('Perplexity vs Generation Diversity — Embed Dim Comparison', fontsize=13, fontweight='bold')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

---
## Plot ME11 — Overlap & Repetition vs Temperature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for ax, metric in zip(axes, ['Overlap', 'Repetition']):
    for label, df in medata.items():
        ax.plot(df['Temperature'], df[metric], marker='o', label=label,
                color=MECOLORS[label], linewidth=2, markersize=7)
    ax.set_title(f'{metric} vs Temperature  (lower is better)', fontweight='bold')
    ax.set_xlabel('Temperature')
    ax.set_ylabel(metric)
    ax.legend(title='Embed Dim')
fig.suptitle('Coherence Metrics: Overlap & Repetition — Embedding Dims', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
---
# Part 6 — Cross-Axis Eval: Layers vs Embedding Dims
Combining Part 4 (layer eval, E256 fixed) and Part 5 (embed dim eval, L4 fixed).

---
## Plot CE1 — Perplexity Scaling: Layers vs Embedding Dim

In [ ]:
layer_nums_eval = [1, 2, 4, 8]
layer_perp = [mdata[l]['Perplexity'].iloc[0] for l in ['L1','L2','L4','L8']]
edim_nums_eval = [128, 256, 512, 1024]
edim_perp  = [medata[l]['Perplexity'].iloc[0] for l in MELABELS]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bars = ax.bar([str(l) for l in layer_nums_eval], layer_perp,
              color=['#e63946','#f4a261','#2a9d8f','#457b9d'], alpha=0.88)
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=9, fontweight='bold')
ax.set_title('Perplexity vs Num Layers\n(E256 fixed)', fontweight='bold')
ax.set_xlabel('Num Layers')
ax.set_ylabel('Perplexity (lower is better)')

ax = axes[1]
bars2 = ax.bar([str(d) for d in edim_nums_eval], edim_perp,
               color=['#9b2226','#ca6702','#0a9396','#005f73'], alpha=0.88)
ax.bar_label(bars2, fmt='%.1f', padding=4, fontsize=9, fontweight='bold')
ax.set_title('Perplexity vs Embedding Dim\n(L4 fixed)', fontweight='bold')
ax.set_xlabel('Embedding Dim')
ax.set_ylabel('Perplexity (lower is better)')

fig.suptitle('Perplexity Scaling: Layers vs Embedding Dim', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot CE2 — Perplexity Scaling Curves (Log Scale, Both Axes)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(layer_nums_eval, layer_perp, 'o-', color='#2a9d8f', linewidth=2.5,
        markersize=9, label='Vary Layers (E256 fixed)')
ax.plot(edim_nums_eval, edim_perp, 's--', color='#e63946', linewidth=2.5,
        markersize=9, label='Vary Embed Dim (L4 fixed)')

for x, y in zip(layer_nums_eval, layer_perp):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8, color='#2a9d8f')
for x, y in zip(edim_nums_eval, edim_perp):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, -16),
                ha='center', fontsize=8, color='#e63946')

ax.set_xscale('log', base=2)
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.set_title('Perplexity Scaling: Rate of Improvement (log2 x-axis)', fontsize=13, fontweight='bold')
ax.set_xlabel('Parameter Count (Layers or Embed Dim, log2)')
ax.set_ylabel('Perplexity')
ax.legend()
plt.tight_layout()
plt.show()

---
## Plot CE3 — Composite Quality Score at Best Temperature: Layers vs Embed Dim

In [ ]:
layer_scores = {
    l: composite_score(mdata[l].iloc[mdata[l].apply(composite_score, axis=1).idxmax()])
    for l in ['L1','L2','L4','L8']
}
edim_scores = {
    l: composite_score(medata[l].iloc[medata[l].apply(composite_score, axis=1).idxmax()])
    for l in MELABELS
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
lkeys = ['L1','L2','L4','L8']
lvals = [layer_scores[l] for l in lkeys]
bars = ax.bar(lkeys, lvals, color=['#e63946','#f4a261','#2a9d8f','#457b9d'], alpha=0.88)
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9, fontweight='bold')
ax.set_ylim(min(lvals)*0.97, max(lvals)*1.03)
ax.set_title('Composite Score vs Num Layers\n(E256 fixed)', fontweight='bold')
ax.set_xlabel('Num Layers')
ax.set_ylabel('Composite Score (higher is better)')

ax = axes[1]
evals_scores = [edim_scores[l] for l in MELABELS]
bars2 = ax.bar(MELABELS, evals_scores, color=['#9b2226','#ca6702','#0a9396','#005f73'], alpha=0.88)
ax.bar_label(bars2, fmt='%.4f', padding=4, fontsize=9, fontweight='bold')
ax.set_ylim(min(evals_scores)*0.97, max(evals_scores)*1.03)
ax.set_title('Composite Score vs Embed Dim\n(L4 fixed)', fontweight='bold')
ax.set_xlabel('Embedding Dim')
ax.set_ylabel('Composite Score (higher is better)')

fig.suptitle('Composite Quality Score at Best Temperature: Layers vs Embed Dim',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot CE4 — All Metrics at T=1.0: Layers vs Embed Dim (Grouped Bar)

In [ ]:
FIXED_TEMP = 1.0
compare_metrics = ['Distinct_1', 'Distinct_2', 'Overlap', 'Repetition']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Layers panel
ax = axes[0]
lkeys = ['L1','L2','L4','L8']
lcolors = ['#e63946','#f4a261','#2a9d8f','#457b9d']
x = np.arange(len(compare_metrics))
w = 0.18
for i, lk in enumerate(lkeys):
    row = mdata[lk][mdata[lk]['Temperature'] == FIXED_TEMP].iloc[0]
    ax.bar(x + i*w, [row[m] for m in compare_metrics], w, label=lk,
           color=lcolors[i], alpha=0.88)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(compare_metrics)
ax.set_title(f'Metrics at T={FIXED_TEMP} — Vary Layers (E256)', fontweight='bold')
ax.set_ylabel('Value')
ax.legend(title='Layers', fontsize=9)

# Embed dim panel
ax = axes[1]
ecolors = ['#9b2226','#ca6702','#0a9396','#005f73']
for i, ek in enumerate(MELABELS):
    row = medata[ek][medata[ek]['Temperature'] == FIXED_TEMP].iloc[0]
    ax.bar(x + i*w, [row[m] for m in compare_metrics], w, label=ek,
           color=ecolors[i], alpha=0.88)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(compare_metrics)
ax.set_title(f'Metrics at T={FIXED_TEMP} — Vary Embed Dim (L4)', fontweight='bold')
ax.set_ylabel('Value')
ax.legend(title='Embed Dim', fontsize=9)

fig.suptitle(f'All Metrics at T={FIXED_TEMP}: Layers vs Embedding Dim', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Plot CE5 — Perplexity vs Composite Score: All Configs on One Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Layer configs (squares)
lkeys = ['L1','L2','L4','L8']
lcolors = ['#e63946','#f4a261','#2a9d8f','#457b9d']
for lk, lc in zip(lkeys, lcolors):
    px = mdata[lk]['Perplexity'].iloc[0]
    py = layer_scores[lk]
    ax.scatter(px, py, color=lc, marker='s', s=160, zorder=4, edgecolors='white', linewidths=1.5)
    ax.annotate(lk, (px, py), textcoords='offset points', xytext=(7, 5),
                fontsize=10, fontweight='bold', color=lc)

# Embed dim configs (circles)
ecolors = ['#9b2226','#ca6702','#0a9396','#005f73']
for ek, ec in zip(MELABELS, ecolors):
    px = medata[ek]['Perplexity'].iloc[0]
    py = edim_scores[ek]
    ax.scatter(px, py, color=ec, marker='o', s=160, zorder=4, edgecolors='white', linewidths=1.5)
    ax.annotate(ek, (px, py), textcoords='offset points', xytext=(7, 5),
                fontsize=10, fontweight='bold', color=ec)

# Legend
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0],[0], marker='s', color='w', markerfacecolor='grey', markersize=10, label='Vary Layers (E256)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', markersize=10, label='Vary Embed Dim (L4)'),
]
ax.legend(handles=legend_handles)
ax.invert_xaxis()
ax.set_xlabel('Perplexity  (lower is better ←)')
ax.set_ylabel('Composite Quality Score  (higher is better ↑)')
ax.set_title('Perplexity vs Composite Score — All Configs\n(ideal = top-left)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()